# 第2回 宿題（発展）：用語集を LoRA で学習し、学習前後を比較する

講義の `lora_format_tuning.py`（Colab用ノートブック）をベースにした発展用ノートブックです。
**モデル、revision、LoRAの基本設定、SFTTrainer、貪欲生成による前後比較**は元の演習を引き継ぎます。

- 学習データ：`glossary_sft.jsonl`（235件、50用語）
- 評価データ：`glossary_eval.jsonl`（63件、A：学習済み50問／B：未学習13問）
- 学習データの回答は **定義文そのもの**。FAQのようなJSON形式の出力ではないため、JSON形式遵守率・`needs_review` の判定は用いません。
- **自動指標**：正解文と生成文の *文字2-gram F1*。意味的な正解率ではありません。**主な評価**は最後に目視で0/1/2点をつけます。
- Bは「LoRAに定義を直接学習させていない用語」であり、「事前学習モデルが知らない用語」を意味しません。

セルは **0 → 7の順** に実行します。元演習の4bit量子化（セクション8）は今回の主目的と異なるため省略しました。

## 0. 環境の確認とライブラリの導入
元演習で指定されたバージョンを利用します。Colabのランタイムは **T4 GPU** を選択してください。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True, text=True,
    ).stdout
except FileNotFoundError:
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# 元の演習と同じバージョン（torch は Colab 同梱のものを利用）
%pip uninstall -y -q torchao
%pip install -q "transformers==5.18.0" "peft==0.21.1" "trl==1.14.2" "accelerate==1.15.0" "datasets==5.1.0"

import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

元の `Qwen/Qwen3.5-0.8B`、revision、学習率、LoRAランクなどは維持します。
**14件×20エポックから235件×2エポック**に変更します。バッチ4・勾配累積2なら、概ね **60 optimizer steps** になります（元は40 steps）。定義文が長いため生成上限を256トークンに変更します。

In [3]:
from pathlib import Path
import os, json, time, shutil

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 2,  # 235件なので元の20から変更
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,  # 定義文を途中で切りにくくする
    "gen": {"do_sample": False},  # 貪欲生成。前後で固定
    "work_dir": "/content/glossary_lora_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lora_lab",
}
WORK = Path(CONFIG["work_dir"])
WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # TrueでDriveにも結果を保存
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 2,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/glossary_lora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lora_lab"
}


## 2. JSONLの読み込み

Codespacesで作成した **`glossary_sft.jsonl`** と **`glossary_eval.jsonl`** をColabにアップロードします。

学習用JSONLは `messages` のみ、評価用JSONLには `messages`、`group`、`source`、`term_id` があります。評価用の正解定義は採点のみに使い、モデルへの入力には渡しません。

In [4]:
from google.colab import files
files.upload()  # glossary_sft.jsonl と glossary_eval.jsonl を2つ選ぶ

Saving glossary_sft.jsonl to glossary_sft.jsonl
Saving glossary_eval.jsonl to glossary_eval.jsonl


{'glossary_sft.jsonl': b'{"messages": [{"role": "user", "content": "\xe3\x80\x8cAI\xe3\x82\xa8\xe3\x83\xb3\xe3\x82\xb8\xe3\x83\x8b\xe3\x82\xa2\xe3\x83\xaa\xe3\x83\xb3\xe3\x82\xb0\xe3\x80\x8d\xe3\x81\xa8\xe3\x81\xaf\xe4\xbd\x95\xe3\x81\xa7\xe3\x81\x99\xe3\x81\x8b\xef\xbc\x9f"}, {"role": "assistant", "content": "\xe5\xad\xa6\xe7\xbf\x92\xe6\xb8\x88\xe3\x81\xbf\xe3\x81\xae\xe5\x9f\xba\xe7\x9b\xa4\xe3\x83\xa2\xe3\x83\x87\xe3\x83\xab\xef\xbc\x88\xe4\xb8\xbb\xe3\x81\xabLLM\xef\xbc\x89\xe3\x82\x92\xe9\x83\xa8\xe5\x93\x81\xe3\x81\xa8\xe3\x81\x97\xe3\x81\xa6\xe4\xbd\xbf\xe3\x81\x84\xe3\x80\x81\xe5\xae\x9f\xe9\x9a\x9b\xe3\x81\xab\xe5\xbd\xb9\xe7\xab\x8b\xe3\x81\xa4\xe3\x82\xa2\xe3\x83\x97\xe3\x83\xaa\xe3\x82\xb1\xe3\x83\xbc\xe3\x82\xb7\xe3\x83\xa7\xe3\x83\xb3\xe3\x82\x84\xe3\x82\xb7\xe3\x82\xb9\xe3\x83\x86\xe3\x83\xa0\xe3\x82\x92\xe8\xa8\xad\xe8\xa8\x88\xe3\x83\xbb\xe6\xa7\x8b\xe7\xaf\x89\xe3\x83\xbb\xe8\xa9\x95\xe4\xbe\xa1\xe3\x83\xbb\xe9\x81\x8b\xe7\x94\xa8\xe3\x81\x99\xe3\x82\x8b\xe6\x8a\x80\

In [5]:
from collections import Counter

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

for name in ("glossary_sft.jsonl", "glossary_eval.jsonl"):
    if not Path(name).exists():
        raise FileNotFoundError(f"{name} をアップロードしてください")
    shutil.copyfile(name, WORK / name)

train_rows = read_jsonl("glossary_sft.jsonl")
eval_rows = read_jsonl("glossary_eval.jsonl")

assert all(
    len(r["messages"]) == 2
    and r["messages"][0]["role"] == "user"
    and r["messages"][1]["role"] == "assistant"
    for r in train_rows + eval_rows
), "JSONLの messages 形式を確認してください"
assert all(r.get("group") in ("A", "B") for r in eval_rows), "評価データにgroup A/Bが必要です"
assert all("source" in r and "term_id" in r for r in eval_rows)

train_questions = {r["messages"][0]["content"] for r in train_rows}
eval_questions = {r["messages"][0]["content"] for r in eval_rows}
assert train_questions.isdisjoint(eval_questions), "学習と評価で質問文が重複しています"
assert len(eval_questions) == len(eval_rows), "評価データ内に重複した質問があります"
print("train:", len(train_rows), "test:", len(eval_rows))
print("group:", dict(Counter(r["group"] for r in eval_rows)))
print("学習例:", train_rows[0])
print("評価例:", eval_rows[0])

train: 235 test: 63
group: {'A': 50, 'B': 13}
学習例: {'messages': [{'role': 'user', 'content': '「AIエンジニアリング」とは何ですか？'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}]}
評価例: {'messages': [{'role': 'user', 'content': '初めて学ぶ人にも分かるように「AIエンジニアリング」を説明してください。'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}], 'group': 'A', 'source': 'glossary', 'term_id': 'ai-engineering'}


## 3. モデルの読み込みと学習前評価

元演習と同じ基盤モデルを **32bit・量子化なし** で読み込みます。

評価では質問文のみをモデルに渡し、`assistant` の正解定義は与えません。プロンプトの条件を前後で揃えるため、SFTの学習例にも同じシステム文を追加します。

自動指標の **文字2-gram F1** は、出力と模範定義の文字列上の重なりを測ります。正しい言い換えでも低くなる可能性があるため、正解率と解釈しないでください。

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("この演習はGPU（T4等）で実行してください")
DEVICE = "cuda"
torch.manual_seed(CONFIG["seed"])

tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32
).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [7]:
SYSTEM = (
    "あなたはAI技術の用語を説明するアシスタントです。"
    "質問された用語について、日本語で正確かつ簡潔に説明してください。"
)

def build_messages(example):
    # assistant の模範回答はモデルに渡さない
    return [
        {"role": "system", "content": SYSTEM},
        example["messages"][0],
    ]

def generate(model, example):
    model.eval()
    prompt = tok.apply_chat_template(
        build_messages(example),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # 元演習に合わせる
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            pad_token_id=tok.pad_token_id,
            **CONFIG["gen"],
        )
    return tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

In [8]:
from collections import Counter
import re
import pandas as pd
pd.set_option("display.max_colwidth", 170)

def char_ngram_f1(pred: str, reference: str, n: int = 2) -> float:
    """文字n-gramの重複による近似指標。意味的な正解率ではない。"""
    def grams(s):
        s = re.sub(r"\s+", "", s)
        return Counter(s[i:i+n] for i in range(len(s) - n + 1)) if len(s) >= n else Counter([s])

    a, b = grams(pred), grams(reference)
    if not a or not b:
        return 0.0
    common = sum((a & b).values())
    precision = common / sum(a.values())
    recall = common / sum(b.values())
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0


def summarize(rows, label, seconds):
    summary = {"label": label, "n": len(rows), "seconds": round(seconds, 1)}
    for group in ("A", "B"):
        part = [r for r in rows if r["group"] == group]
        summary[group] = {
            "n": len(part),
            "mean_char2_f1": round(sum(r["char2_f1"] for r in part) / len(part), 4) if part else None,
        }
    return summary


def evaluate(model, label):
    """同じ63問の回答と自動指標を保存。評価の正解定義は入力しない。"""
    rows = []
    t0 = time.time()
    for i, ex in enumerate(eval_rows, start=1):
        question = ex["messages"][0]["content"]
        gold = ex["messages"][1]["content"]
        raw = generate(model, ex)
        rows.append({
            "group": ex["group"],
            "source": ex["source"],
            "term_id": ex["term_id"],
            "question": question,
            "gold": gold,
            "raw": raw,
            "char2_f1": round(char_ngram_f1(raw, gold), 4),
        })
        if i % 10 == 0:
            print(f"{label}: {i}/{len(eval_rows)}")
    summary = summarize(rows, label, time.time() - t0)
    (WORK / f"eval_{label}.json").write_text(
        json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary, rows

In [9]:
before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame(before_rows)[["group", "question", "gold", "raw", "char2_f1"]].head(8))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


before: 10/63
before: 20/63
before: 30/63
before: 40/63
before: 50/63
before: 60/63
{
  "label": "before",
  "n": 63,
  "seconds": 615.0,
  "A": {
    "n": 50,
    "mean_char2_f1": 0.0617
  },
  "B": {
    "n": 13,
    "mean_char2_f1": 0.0477
  }
}


,group,question,gold,raw,char2_f1
0,A,初めて学ぶ人にも分かるように「AIエンジニアリング」を説明してください。,学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。,「AI エンジニアリング」は、**「人工知能（AI）の技術」を「実社会で使えるビジネス・産業で活用するエンジニアリング」**という概念です。\n\n初めて学ぶ人にも分かりやすく、以下の 3 つの視点で説明します。\n\n### 1. 基本定義\nAI エンジニアリングとは、**「AI の仕組み（アルゴリズム、データ、モデルなど）」...,0.0734
1,A,初めて学ぶ人にも分かるように「評価と改善のサイクル」を説明してください。,作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。,「評価と改善のサイクル」は、**「評価（結果）」→「改善（行動）」→「評価（結果）」**という一連の循環プロセスです。\n\n初めて学ぶ人にも分かりやすく、以下の 3 つのステップで説明します。\n\n### 1. 評価（結果）\n**「今、どういった結果が得られたか」**\n* **目的**: 現在の状態を測り、その結果を記...,0.0644
2,A,初めて学ぶ人にも分かるように「LLM（大規模言語モデル）」を説明してください。,大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。,**LLM（大規模言語モデル）**は、**「大規模言語モデル」**の略です。\n\n簡単に言うと、**「人間が話せる言語（日本語など）を、巨大なデータを使って学習した、非常に強力で汎用性の高い AI 言語モデル」**のことです。\n\n主な特徴は以下の通りです：\n\n* **学習データ量**: 人間が話せる言語（日本語など）...,0.1037
3,A,初めて学ぶ人にも分かるように「トークン」を説明してください。,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,「トークン（トークン）」は、AI 技術において**「データや文脈を個別に識別し、その特徴を抽出・分析する単位」**です。\n\n一言で言うと、**「単語（文脈）ではなく、その「意味」を独立して捉える小さなブロック」**です。\n\n### 具体的なイメージ\n想像してみてください：\n* **単語（例：「猫」）**：意味が明確...,0.0531
4,B,初めて学ぶ人にも分かるように「コンテキストウィンドウ」を説明してください。,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,「コンテキストウィンドウ」は、AI 技術（特に大規模言語モデル）で使われる重要な概念です。\n\n簡単に言うと、**「AI が文脈を理解し、その文脈の中で正しい答えを返すための、文脈を提示する小さなウィンドウ（枠組み）」のこと**です。\n\nその意味をより詳しく説明すると、以下の 3 つの点で理解できます。\n\n1. **文...,0.0359
5,A,初めて学ぶ人にも分かるように「プロンプト」を説明してください。,LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。,「プロンプト（プロンプト）」は、AI 技術の用語です。\n\n**定義**\nプロンプトは、AI の「人間が書いた言葉」です。ユーザーが AI に指示を出す際に、この言葉がそのまま使われます。\n\n**使い分け**\n* **プロンプト**: AI に「何を言いたいのか」を指示する言葉。\n* **タスク**: AI に...,0.0000
6,A,初めて学ぶ人にも分かるように「システムプロンプト」を説明してください。,利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。,「システムプロンプト」は、AI 機器（大規模言語モデルなど）が人間が指示する**「指令の文書」**のことです。\n\n初めて学ぶ人にも分かりやすく説明すると、以下のようになります。\n\n* **意味**: 人間が「AI にこう言いたい」という言葉の**書き方**です。\n* **目的**: AI がその文書を読み、必要な...,0.0453
7,A,初めて学ぶ人にも分かるように「temperature」を説明してください。,次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。,「温度」は、物質の分子運動の強さや熱運動の平均速度を表す物理量です。\n\n簡単に言うと、**「熱が強いほど分子が動きやすくなる」**というものです。\n\n* **低温**：分子は動きにくく、静止に近い状態です。\n* **高温**：分子は動きやすくなり、衝突しやすく、熱が強い状態です。\n\n例え、氷が氷水に溶け、熱が...,0.0321


## 4. LoRA 学習

元演習と同じ `LoraConfig`、`SFTConfig`、`SFTTrainer` を用います。学習用データ235件を `messages` 型のデータセットにし、評価時と同じ `SYSTEM` を先頭に追加します。

**変更点**：学習エポック数は2、チェックポイント間隔は30ステップ、学習元を架空FAQから用語集へ変更。学習に `eval_rows` は使いません。

元演習と同様、`SFTTrainer` の標準設定（会話テキスト全体に対する損失）を利用します。assistant部分だけの損失に変更する実験は、必要になった場合に別条件として扱ってください。

In [10]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習時にも評価と同じSYSTEM・質問形式を使う
    return {"messages": build_messages(example) + [example["messages"][1]]}

train_ds = Dataset.from_list([to_chat(ex) for ex in train_rows])
print("学習件数:", len(train_ds))
print("サンプル:", train_ds[0])

model.config.use_cache = False
lora = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
USE_AMP = False  # 元演習と同じく32bitで学習
sft = SFTConfig(
    output_dir=str(WORK / "ckpt"),
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=CONFIG["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=10,
    save_steps=30,
    save_total_limit=2,
    fp16=USE_AMP,
    max_length=CONFIG["max_seq_len"],
    packing=False,
    report_to="none",
    seed=CONFIG["seed"],
)
trainer = SFTTrainer(
    model=model,
    args=sft,
    train_dataset=train_ds,
    peft_config=lora,
    processing_class=tok,
)
trainer.model.print_trainable_parameters()

学習件数: 235
サンプル: {'messages': [{'role': 'system', 'content': 'あなたはAI技術の用語を説明するアシスタントです。質問された用語について、日本語で正確かつ簡潔に説明してください。'}, {'role': 'user', 'content': '「AIエンジニアリング」とは何ですか？'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}]}


Tokenizing train dataset:   0%|          | 0/235 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/235 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/235 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/235 [00:00<?, ? examples/s]

trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


In [11]:
t0 = time.time()
trainer.train()
print(
    f"学習時間: {time.time()-t0:.0f} 秒 | "
    f"ステップ数: {trainer.state.global_step} | "
    f"エポック: {trainer.state.epoch:.1f}"
)
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)

tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # dropout を無効化

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss
10,2.555504
20,1.560360
30,1.210769
40,0.766567
50,0.638022
60,0.586358


学習時間: 194 秒 | ステップ数: 60 | エポック: 2.0


## 5. 学習後評価

学習前と **同じ63問・同じシステムプロンプト・同じ貪欲生成設定** で評価します。学習前の結果はすでにファイルに保存済みです。

In [12]:
after_summary, after_rows = evaluate(tuned, "after")
comparison_summary = pd.DataFrame([
    {"group": g, "n": before_summary[g]["n"],
     "before_char2_f1": before_summary[g]["mean_char2_f1"],
     "after_char2_f1": after_summary[g]["mean_char2_f1"],
     "delta": round(after_summary[g]["mean_char2_f1"] - before_summary[g]["mean_char2_f1"], 4)}
    for g in ("A", "B")
])
display(comparison_summary)

after: 10/63
after: 20/63
after: 30/63
after: 40/63
after: 50/63
after: 60/63
{
  "label": "after",
  "n": 63,
  "seconds": 213.8,
  "A": {
    "n": 50,
    "mean_char2_f1": 0.306
  },
  "B": {
    "n": 13,
    "mean_char2_f1": 0.1031
  }
}


,group,n,before_char2_f1,after_char2_f1,delta
0,A,50,0.0617,0.3060,0.2443
1,B,13,0.0477,0.1031,0.0554


## 6. 回答を比較し、手動で内容を採点する

**手動判定を主指標**とします。形式が違っても内容が正しければ2点にできます。

- **2点**：重要な内容が正しく説明されている
- **1点**：一部は正しいが重要な情報が欠ける
- **0点**：内容が誤っている、意味が通らない、無回答

以下で `comparison_for_scoring.csv` を出力します。Excel等で `score_before`、`score_after` に **0 / 1 / 2** を入力し、`comparison_scored.csv` としてアップロードしてください。採点は同じ基準で行い、学習前後の順序で判定が偏らないように注意してください。

自動指標のF1は参照用です。Bの13問は小さいサンプルなので、平均値の変動だけで一般化性能の改善・悪化を断定しないでください。

In [13]:
paired = []
for b, a in zip(before_rows, after_rows):
    assert b["question"] == a["question"] and b["term_id"] == a["term_id"]
    paired.append({
        "group": b["group"],
        "source": b["source"],
        "term_id": b["term_id"],
        "question": b["question"],
        "gold": b["gold"],
        "before_answer": b["raw"],
        "after_answer": a["raw"],
        "before_char2_f1": b["char2_f1"],
        "after_char2_f1": a["char2_f1"],
        "score_before": "",
        "score_after": "",
    })
comparison = pd.DataFrame(paired)
comparison.to_csv(WORK / "comparison_for_scoring.csv", index=False, encoding="utf-8-sig")
display(comparison.head(5))
print("保存先:", WORK / "comparison_for_scoring.csv")
print("比較表のCSVをダウンロードして、score_before / score_after を0,1,2で記入してください")

,group,source,term_id,question,gold,before_answer,after_answer,before_char2_f1,after_char2_f1,score_before,score_after
0,A,glossary,ai-engineering,初めて学ぶ人にも分かるように「AIエンジニアリング」を説明してください。,学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。,「AI エンジニアリング」は、**「人工知能（AI）の技術」を「実社会で使えるビジネス・産業で活用するエンジニアリング」**という概念です。\n\n初めて学ぶ人にも分かりやすく、以下の 3 つの視点で説明します。\n\n### 1. 基本定義\nAI エンジニアリングとは、**「AI の仕組み（アルゴリズム、データ、モデルなど）」...,AIの各技術（モデル、データ、ツール、評価など）を、モデルの選定、データとツールの選び方、評価と改善のサイクルで組み合わせて使うこと。本講座では「ツールを呼び出すためのAPIキーを公開する」作業を、AIエンジニアリングの例として扱う。,0.0734,0.2650,,
1,A,glossary,evaluation-loop,初めて学ぶ人にも分かるように「評価と改善のサイクル」を説明してください。,作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。,「評価と改善のサイクル」は、**「評価（結果）」→「改善（行動）」→「評価（結果）」**という一連の循環プロセスです。\n\n初めて学ぶ人にも分かりやすく、以下の 3 つのステップで説明します。\n\n### 1. 評価（結果）\n**「今、どういった結果が得られたか」**\n* **目的**: 現在の状態を測り、その結果を記...,モデルの性能を測るためにある評価集（テストデータ）を作る、その結果を見てモデルを改善する、その結果を見て次の評価集を作る、という回を1サイクルと数える。本講座では、各回のコード変更について、テストで確認できるほど短い評価集を作るのが原則。,0.0644,0.2348,,
2,A,glossary,llm,初めて学ぶ人にも分かるように「LLM（大規模言語モデル）」を説明してください。,大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。,**LLM（大規模言語モデル）**は、**「大規模言語モデル」**の略です。\n\n簡単に言うと、**「人間が話せる言語（日本語など）を、巨大なデータを使って学習した、非常に強力で汎用性の高い AI 言語モデル」**のことです。\n\n主な特徴は以下の通りです：\n\n* **学習データ量**: 人間が話せる言語（日本語など）...,大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、コード生成、翻訳など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。,0.1037,0.9518,,
3,A,glossary,token,初めて学ぶ人にも分かるように「トークン」を説明してください。,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,「トークン（トークン）」は、AI 技術において**「データや文脈を個別に識別し、その特徴を抽出・分析する単位」**です。\n\n一言で言うと、**「単語（文脈）ではなく、その「意味」を独立して捉える小さなブロック」**です。\n\n### 具体的なイメージ\n想像してみてください：\n* **単語（例：「猫」）**：意味が明確...,LLMがテキストを扱う最小単位。単語や文字ではなく、単語の英単語や、ハイパーパラメータの値、引数の型などによって変わる。APIのレート制限、メモリ使用量、モデルの選択、出力の形式（JSON、テキストエディタなど）を決めるために使う。,0.0531,0.3014,,
4,B,glossary,context-window,初めて学ぶ人にも分かるように「コンテキストウィンドウ」を説明してください。,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,「コンテキストウィンドウ」は、AI 技術（特に大規模言語モデル）で使われる重要な概念です。\n\n簡単に言うと、**「AI が文脈を理解し、その文脈の中で正しい答えを返すための、文脈を提示する小さなウィンドウ（枠組み）」のこと**です。\n\nその意味をより詳しく説明すると、以下の 3 つの点で理解できます。\n\n1. **文...,LLMが読み込むテキストの範囲。質問する文章の前後に挟むように設定する。本講座では質問する文章の左側と右側を同じ長さの文章に置き換える。,0.0359,0.0694,,


保存先: /content/glossary_lora_lab/comparison_for_scoring.csv
比較表のCSVをダウンロードして、score_before / score_after を0,1,2で記入してください


In [14]:
# 手動採点後にこのセルを再実行。ファイルはColab左側のファイル欄からアップロードできます。
# 必要なら以下の2行のコメントを外してファイル選択ダイアログを表示してください。
# from google.colab import files
# files.upload()  # comparison_scored.csv を選択

scored_path = WORK / "comparison_scored.csv"
if Path("comparison_scored.csv").exists():
    shutil.copy2("comparison_scored.csv", scored_path)

if not scored_path.exists():
    print("採点後の comparison_scored.csv はまだありません。先にCSVに0/1/2点を記入してください。")
else:
    scored = pd.read_csv(scored_path)
    for col in ("score_before", "score_after"):
        scored[col] = pd.to_numeric(scored[col], errors="raise")
        assert scored[col].notna().all() and scored[col].isin([0, 1, 2]).all(), f"{col} は0,1,2のみ"
    score_summary = (
        scored.groupby("group")[["score_before", "score_after"]]
        .mean().mul(50).round(1)
    )
    score_summary["delta_pt"] = score_summary["score_after"] - score_summary["score_before"]
    display(score_summary)
    print("各スコアは平均点/2を百分率表示したものです。厳密な正解率ではありません。")

採点後の comparison_scored.csv はまだありません。先にCSVに0/1/2点を記入してください。


## 7. 結果の保存と提出

提出に役立つのは、学習条件を記録した `result.json`、学習前後の `eval_*.json`、回答比較表 `comparison_for_scoring.csv` です。これらは `/content/glossary_lora_lab/` に保存されます。Colabはセッション終了で消えるため、必要に応じてダウンロードしてください。

In [15]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in (transformers, peft, trl, torch)},
    "method": "LoRA（量子化なし、用語集の定義文でSFT）",
    "train_size": len(train_rows),
    "eval_size": len(eval_rows),
    "eval_group_sizes": dict(Counter(r["group"] for r in eval_rows)),
    "base_model_memory_gb": BASE_MEMORY_GB,
    "training_steps": trainer.state.global_step,
    "training_epochs": trainer.state.epoch,
    "before": before_summary,
    "after": after_summary,
    "note": "char2_f1 は文字列類似度であり意味的正解率ではない。正確性は別途目視採点する。",
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
if USE_DRIVE:
    for name in [
        "result.json", "eval_before.json", "eval_after.json",
        "glossary_sft.jsonl", "glossary_eval.jsonl", "comparison_for_scoring.csv"
    ]:
        shutil.copy2(WORK / name, Path(CONFIG["drive_dir"]) / name)
print("結果ファイル:")
for path in sorted(WORK.glob("*.json*")):
    print(" -", path)
print(" -", WORK / "comparison_for_scoring.csv")

結果ファイル:
 - /content/glossary_lora_lab/eval_after.json
 - /content/glossary_lora_lab/eval_before.json
 - /content/glossary_lora_lab/glossary_eval.jsonl
 - /content/glossary_lora_lab/glossary_sft.jsonl
 - /content/glossary_lora_lab/result.json
 - /content/glossary_lora_lab/comparison_for_scoring.csv


In [16]:
# 必要な場合に実行：評価結果をZIPでダウンロード
import zipfile
archive = WORK / "glossary_lora_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["result.json", "eval_before.json", "eval_after.json", "comparison_for_scoring.csv"]:
        z.write(WORK / name, arcname=name)
from google.colab import files
files.download(str(archive))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>